## Generacja tekstu z użyciem prostego transformera

In [ ]:
! pip install keras_nlp sentencepiece

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_nlp
import numpy as np
import random
from tensorflow.keras.layers import TextVectorization

Wczytanie danych do zmiennych

In [ ]:
crime_and_punishment_url = 'https://www.gutenberg.org/files/2554/2554-0.txt'
brothers_of_karamazov_url = 'https://www.gutenberg.org/files/28054/28054-0.txt'
the_idiot_url = 'https://www.gutenberg.org/files/2638/2638-0.txt'

paths = [crime_and_punishment_url, brothers_of_karamazov_url, the_idiot_url]
names = ['Crime and Punishment', 'Brothers of Karamazov', 'The Idiot']
texts = ''
for index, path in enumerate(paths):
    filepath = keras.utils.get_file(f'{names[index]}.txt', origin=path)
    text = ''
    with open(filepath, encoding='utf-8') as f:
        text = f.read()
        # Pomijamy przedmowe i wstęp z ksiązki
        texts += text[10000:]

In [ ]:
# przykładowe 500 znaków
texts[25000:25500]

In [ ]:
text_list = texts.split('.')
len(text_list) # 50835 - estymacja ilości zdań

Podział danych na treningowe, testowe i walidacyjne

In [ ]:
random.shuffle(text_list)
length = len(text_list)
text_train = text_list[:int(0.7*length)]
text_test = text_list[int(0.7*length):int(0.85*length)]
text_valid = text_list[int(0.85*length):]

Definicja tworzenia sekwencji i vectorizera

In [ ]:
len(max(text_list).split(' '))

In [ ]:
def custom_standardization(input_string):
    sentence = tf.strings.lower(input_string)  # konwersja całego tekstu na małe litery (normalizacja)
    sentence = tf.strings.regex_replace(sentence, "\n", " ")  # zamiana znaków nowej linii na spacje
    return sentence  # zwrot przetworzonego tekstu

# określenie maksymalnej długości zdania w zbiorze danych
maxlen = len(max(text_list).split(' '))  # wybiera najdłuższe zdanie i liczy jego tokeny (tu: słowa), np. 25

vectorize_layer = TextVectorization(
    standardize=custom_standardization,  # własna funkcja preprocessingu tekstu
    output_mode="int",  # każde słowo zamieniane na indeks liczbowy (tokenizacja)
    output_sequence_length=maxlen+1,  # wszystkie sekwencje mają stałą długość (padding/truncation)
)

vectorize_layer.adapt(text_list)  # budowa słownika (vocabulary) na podstawie danych treningowych
vocab = vectorize_layer.get_vocabulary()  # pobranie listy tokenów (słownik modelu)

Ustalenie wielkości słownika

In [ ]:
vocab_size = len(vocab)
vocab_size # 40677

In [ ]:
index_lookup = dict(zip(range(len(vocab)), vocab))
index_lookup[5] 

Przykładowy wektor reprezentujący zdanie

In [ ]:
v = vectorize_layer(['The final goal of life is...'])
print(v, '\n', len(v[0]))

Stworzenie sekwencji wejściowych z podziałem na zbiór treningowy, walidacyjny i testowy

In [ ]:
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices(text_train)
train_dataset = train_dataset.shuffle(buffer_size=256)
train_dataset = train_dataset.batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices(text_test)
test_dataset = test_dataset.shuffle(buffer_size=256)
test_dataset = test_dataset.batch(batch_size)

valid_dataset = tf.data.Dataset.from_tensor_slices(text_valid)
valid_dataset = valid_dataset.shuffle(buffer_size=256)
valid_dataset = valid_dataset.batch(batch_size)

Przygotowanie danych do treningu wraz z optymalizacją przetwarzania kolejnych batchy z przez równoległe pobieranie z wyprzedzeniem w Tensorflow (https://www.tensorflow.org/guide/data_performance?hl=pl#prefetching)

In [ ]:
def preprocess_text(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_dataset = train_dataset.map(preprocess_text)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

test_dataset = test_dataset.map(preprocess_text)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

valid_dataset = valid_dataset.map(preprocess_text)
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)

Przykład przeprocesowanego tekstu

In [ ]:
for entry in train_dataset.take(1):
    print(entry)

### Model Transformer do generowania tekstu

Poniższy model implementuje architekturę typu **Transformer (decoder-only)** do zadania modelowania języka, czyli przewidywania kolejnych tokenów w sekwencji.

#### Parametry modelu

- `embed_dim = 128` – wymiar embeddingu (reprezentacji wektorowej tokenów),
- `num_heads = 4` – liczba głów w mechanizmie multi-head attention.
- Wejście to sekwencja tokenów o długości maxlen.
- Każdy token jest zakodowany jako indeks słownika.
- self-attention pozwala modelowi analizować zależności między tokenami,
- multi-head attention umożliwia uchwycenie różnych relacji równolegle,

---

In [ ]:
embed_dim = 128
num_heads = 4

def create_model():
    inputs = keras.layers.Input(shape=(maxlen,), dtype=tf.int32)
    embedding_layer = keras_nlp.layers.TokenAndPositionEmbedding(vocab_size, maxlen, embed_dim)(inputs)
    # Dodajemy informację o pozycji w sekwencji - w odróżnieniu od poprzednich zajęć ta warstwa koduje też kolejność tokenów
    decoder = keras_nlp.layers.TransformerDecoder(intermediate_dim=embed_dim,
                                                  num_heads=num_heads,
                                                  dropout=0.5)(embedding_layer)

    outputs = keras.layers.Dense(vocab_size, activation='softmax')(decoder)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer="adam",
        loss='sparse_categorical_crossentropy',
        metrics=[keras_nlp.metrics.Perplexity(), 'accuracy']
    )
    return model

model = create_model()
model.summary()

#### Callback służy do generowania przykładowych tekstów po każdej epoce treningu, co pozwala ocenić jakość modelu.
- Wybierane jest 5 najbardziej prawdopodobnych tokenów,
- Softmax przekształca logity w rozkład prawdopodobieństwa,
- Losowanie z tego rozkładu: zwiększa różnorodność generacji.

In [ ]:
class TextSampler(keras.callbacks.Callback):
    def __init__(self, start_prompt, max_tokens):
        # start_prompt – początkowy tekst (seed), od którego zaczynamy generowanie
        # max_tokens – maksymalna liczba tokenów do wygenerowania
        self.start_prompt = start_prompt
        self.max_tokens = max_tokens

    # Próbkowanie kolejnego tokenu z top-5 najbardziej prawdopodobnych
    def sample_token(self, logits):
        # Wybierz 5 największych wartości (logitów) oraz ich indeksy
        logits, indices = tf.math.top_k(logits, k=5, sorted=True)

        indices = np.asarray(indices).astype("int32")

        # Softmax → zamiana logitów na rozkład prawdopodobieństwa
        preds = keras.activations.softmax(tf.expand_dims(logits, 0))[0]
        preds = np.asarray(preds).astype("float32")

        # Losowanie jednego tokenu zgodnie z rozkładem prawdopodobieństwa
        return np.random.choice(indices, p=preds)

    def on_epoch_end(self, epoch, logs=None):
        # Zaczynamy od tekstu początkowego
        decoded_sample = self.start_prompt

        # Generujemy kolejne tokeny aż do osiągnięcia max_tokens
        for i in range(self.max_tokens - 1):

            # Tokenizacja aktualnego tekstu (obcięcie ostatniego tokenu – shift)
            tokenized_prompt = vectorize_layer([decoded_sample])[:, :-1]

            # Predykcja modelu: rozkład prawdopodobieństwa dla każdego kroku sekwencji
            predictions = self.model.predict([tokenized_prompt], verbose=0)

            # Indeks ostatniego tokenu w sekwencji
            sample_index = len(decoded_sample.strip().split()) - 1

            # Wybór kolejnego tokenu (sampling)
            sampled_token = self.sample_token(predictions[0][sample_index])

            # Zamiana indeksu tokenu na słowo
            sampled_token = index_lookup[sampled_token]

            # Dodanie nowego tokenu do wygenerowanego tekstu
            decoded_sample += " " + sampled_token

        # Wyświetlenie wygenerowanego tekstu po epoce
        print(f"\nSample text:\n{decoded_sample}...\n")


# Pierwsze 4 słowa losowego zdania jako seed (prompt początkowy)
random_sentence = ' '.join(
    random.choice(text_valid).replace('\n', ' ').split(' ')[:4]
)

sampler = TextSampler(random_sentence, maxlen)

# Callback do redukcji learning rate przy braku poprawy
reducelr = keras.callbacks.ReduceLROnPlateau(
    patience=10,
    monitor='val_loss'
)

Trening modelu

In [ ]:
model = create_model()
history = model.fit(train_dataset,
                    validation_data=valid_dataset,
                    epochs=3,
                    callbacks=[sampler, reducelr])

Funkcja pomocnicza do generacji tekstu

In [ ]:
def sample_token(logits):
    """
    Funkcja realizuje tzw. Top-k sampling (k=5), czyli losowanie kolejnego tokenu
    spośród kilku najbardziej prawdopodobnych zamiast wyboru maksimum (argmax).
    """
    # Wybieramy 5 najwyższych logitów (najbardziej prawdopodobne tokeny).
    logits, indices = tf.math.top_k(logits, k=5, sorted=True)
    indices = np.asarray(indices).astype("int32")
    # Przekształcamy je do rozkładu prawdopodobieństwa (softmax).
    preds = keras.activations.softmax(tf.expand_dims(logits, 0))[0]
    preds = np.asarray(preds).astype("float32")
    Losujemy jeden token zgodnie z tym rozkładem.
    return np.random.choice(indices, p=preds)


def generate_text(prompt, response_length=20):
    """
    Funkcja generuje tekst w sposób autoregresyjny na podstawie wytrenowanego modelu.
    """
    decoded_sample = prompt

    for i in range(response_length - 1):
        # Tokenizacja aktualnego tekstu (przesunięcie o 1 – teacher forcing)
        tokenized_prompt = vectorize_layer([decoded_sample])[:, :-1]

        # Predykcja modelu: rozkład prawdopodobieństwa dla każdego kroku
        predictions = model.predict([tokenized_prompt], verbose=0)

        # Indeks ostatniego tokenu w aktualnej sekwencji
        sample_index = len(decoded_sample.strip().split()) - 1

        # Próbkowanie kolejnego tokenu
        sampled_token = sample_token(predictions[0][sample_index])

        # Zamiana indeksu na token (słowo)
        sampled_token = index_lookup[sampled_token]

        # Dodanie tokenu do tekstu
        decoded_sample += " " + sampled_token

    return decoded_sample

Przykład generacji tekstu z użyciem transformera i sekwencji początkowej

In [ ]:
generate_text("Last summer, ")


Z poniższych materiałów można dowiedzieć się więcej o transformerach:
- Artykuł o mechaniźmie atencji (~ 175k cites): https://arxiv.org/abs/1706.03762
- https://www.youtube.com/watch?v=KJtZARuO3JY&ab_channel=GrantSanderson
